# Revisi - Tahap 1: Analisis & Perbaikan Data Test

**Masalah:** metrik model sangat rendah. Dugaan *garbage-in-garbage-out* -> sebagian
besar penyebab BUKAN modelnya, tapi **gold label test yang bermasalah**.

**Apa yang perlu diperbaiki (temuan inspeksi):**
1. **Label noise** - solusi (`cara`) bawaan dataset sendiri sering tak setuju dengan
   `jawaban` (gold). Kalau dataset saja tak konsisten, akurasi jadi plafon palsu.
2. **Soal tak-terjawab** - numglue sering menyebut *laporan/tabel/teks* yang TIDAK
   disertakan di soal -> mustahil dijawab tanpa passage.
3. **Mojibake / soal rusak** - encoding rusak, soal kosong/terlalu pendek.
4. **Jawaban teks-bebas** (easy) - gold kalimat panjang, sulit dinilai otomatis.
5. **Grader false-negative** (sudah diperbaiki di `src/eval/answer_check.py`):
   `\frac` vs `/`, prefix pilihan ganda `A.`, satuan `135 dolar`, tuple `(3,7)`.

Notebook ini: **analisis -> perbaiki -> verifikasi**, dengan output tiap tahap.
Komparator yang dipakai SAMA dengan saat eval, jadi keputusan buang konsisten.

In [1]:
import os, sys
# pindah ke root repo (folder yang punya 'src') supaya import konsisten
p = os.getcwd()
while not os.path.isdir(os.path.join(p, 'src')) and os.path.dirname(p) != p:
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)

repo root: D:\Main Storage\Vscode\FP_NLP\FP_NLP


## Tahap 1 - Analisis data MENTAH

`self-consistency` = seberapa sering `cara` dataset == `jawaban` gold (pakai grader
terbaru). Rendah = label noise = kira-kira **plafon** akurasi yang bisa dicapai model.

In [2]:
from src.preprop.inspect_data import _read_jsonl, inspect_gold

FILES = {
    "numglue": "data/sft/test/numglue_test.jsonl",
    "easy":    "data/sft/test/easy_test.jsonl",
}
raw = {k: _read_jsonl(v) for k, v in FILES.items()}

for k, rows in raw.items():
    s, _ = inspect_gold(rows)
    print(f"[{k}] n={s['n']}  self-consistency(cara==gold) = {s['self_consistency']:.1%}")
    print(f"   bentuk_jawaban: {s['shapes']}")
    print("   masalah terdeteksi:")
    for fl, cnt in s['flags'].items():
        print(f"      {cnt:>4} ({cnt/s['n']:5.1%})  {fl}")
    print()

[numglue] n=300  self-consistency(cara==gold) = 53.7%
   bentuk_jawaban: {'numeric': 300}
   masalah terdeteksi:
       139 (46.3%)  cara_vs_gold_mismatch
         1 ( 0.3%)  butuh_konteks_luar



[easy] n=300  self-consistency(cara==gold) = 32.7%
   bentuk_jawaban: {'expr': 110, 'numeric': 101, 'text': 76, 'fraction': 13}
   masalah terdeteksi:
       202 (67.3%)  cara_vs_gold_mismatch
        76 (25.3%)  jawaban_teks_bebas
         4 ( 1.3%)  soal_pendek
         3 ( 1.0%)  mojibake
         3 ( 1.0%)  butuh_konteks_luar



### Bukti label noise (contoh numglue)
`cara` dataset menyimpulkan jawaban yang berbeda dari `jawaban` gold:

In [3]:
from src.preprop.clean_testset import drop_reason
from src.preprop.inspect_data import answer_from_cara

shown = 0
for r in raw['numglue']:
    if drop_reason(r) == "cara_vs_gold_mismatch":
        print("SOAL :", r['soal'][:110])
        print("  gold      :", r['jawaban'])
        print("  cara bilang:", answer_from_cara(r['cara']))
        print()
        shown += 1
        if shown == 4:
            break

SOAL : Berapa banyak sekolah pemerintah yang ada pada tahun 1920 dibandingkan dengan tahun 1890?
  gold      : 15
  cara bilang: 1

SOAL : Berapa poin persentase tingkat melek huruf laki-laki meningkat antara 1991 dan 2011?
  gold      : 25.52
  cara bilang: 150\%

SOAL : Pada Olimpiade Musim Dingin 2014, berapa banyak acara dari 12 yang tidak mereka menangkan?
  gold      : 4
  cara bilang: 8

SOAL : Berapa banyak orang yang tinggal di kota Indianapolis di bawah usia 18 tahun yang hidup di bawah garis kemiski
  gold      : 19.1
  cara bilang: 200.000



## Tahap 2 - Perbaikan (cleaning)

Buang baris yang bikin akurasi jadi artefak: `jawaban_kosong`, `soal_rusak`,
`mojibake`, `butuh_konteks_luar`, `cara_vs_gold_mismatch`. Sisanya = subset yang
**adil dinilai otomatis**. (Knob: `drop_text=True` untuk buang gold kalimat-bebas.)

In [4]:
from src.preprop.clean_testset import clean, _write_jsonl

clean_rows = {}
for k, rows in raw.items():
    kept, dropped, rep = clean(rows)          # drop_mismatch=True (default)
    clean_rows[k] = kept
    _write_jsonl(kept, f"data/sft/test_clean/{k}_test.jsonl")
    print(f"[{k}] {rep['n_before']} -> {rep['n_after']} disimpan ({rep['keep_rate']:.1%})")
    for reason, cnt in rep['dropped_by_reason'].items():
        print(f"      - {cnt:>4}  {reason}")
    print(f"      -> data/sft/test_clean/{k}_test.jsonl")
    print()

[numglue] 300 -> 161 disimpan (53.7%)
      -  138  cara_vs_gold_mismatch
      -    1  butuh_konteks_luar
      -> data/sft/test_clean/numglue_test.jsonl



[easy] 300 -> 94 disimpan (31.3%)
      -  196  cara_vs_gold_mismatch
      -    4  soal_rusak
      -    3  mojibake
      -    3  butuh_konteks_luar
      -> data/sft/test_clean/easy_test.jsonl



## Tahap 3 - Verifikasi (sesudah vs sebelum)

Setelah baris tak-konsisten dibuang, self-consistency subset bersih harus naik
tajam -> gold yang tersisa bisa dipercaya untuk grading.

In [5]:
print(f"{'set':8} {'self-consistency':>22}   {'n (before->after)':>18}")
for k in FILES:
    sb, _ = inspect_gold(raw[k])
    sa, _ = inspect_gold(clean_rows[k])
    sc_b = sb['self_consistency'] or 0
    sc_a = sa['self_consistency'] or 0
    print(f"{k:8} {sc_b:>9.1%} -> {sc_a:>7.1%}      {sb['n']:>6} -> {sa['n']}")

set            self-consistency    n (before->after)


numglue      53.7% ->  100.0%         300 -> 161


easy         32.7% ->  100.0%         300 -> 94


## Ringkasan - yang sudah DIPROSES

- **Grader diperbaiki** (`answer_check.py`): `\frac`/`/`, prefix `A.`/`B)`, satuan,
  prefix var `A' =`, tuple `(3,7)` aman. 22/22 test lolos (sudah di `main`).
- **Test set dibersihkan** (`clean_testset.py`): buang label-noise + unanswerable +
  mojibake -> `data/sft/test_clean/*.jsonl`.
- **numglue**: ~46% baris label-noise/unanswerable dibuang -> subset bersih ~50%.
- **easy**: drop berat (gold teks-bebas + ekstraksi `cara` lossy). **Caveat:** sebagian
  drop easy mungkin over-drop karena `cara` easy tak pakai `\boxed{}` -> angka diambil
  heuristik. Untuk easy, andalkan grader baru + cek manual subset.

**Lanjut:** buka `02_akurasi_baru.ipynb` untuk hitung akurasi baru pakai test bersih.